# Blocking Factorial Experiments

A full factorial may not fit into one batch, day, or session. Splitting it into smaller blocks can protect important comparisons from nuisance variation, but the split can make some treatment effects indistinguishable from block effects.

The goal is to **choose what information is sacrificed**, rather than let scheduling accidentally confound an important factor. The sign-column calculations build on [Two-Level Factorial Designs](./Two-Level-Factorials.ipynb).

## Complete Blocks vs. Confounded Blocks

| Layout | What each block contains | Consequence |
|---|---|---|
| Complete factorial block | Every treatment combination. | Treatment and block effects can be separated under an additive block model. |
| Fraction of the factorial per block | A selected subset of combinations. | Some factorial contrasts are confounded with block contrasts. |

Here, every factorial treatment combination is still run somewhere in the experiment. This differs from a fractional factorial, where some combinations are not run at all.

**Confounded / aliased columns:** two model columns are identical or differ only by sign over the observed design. Their separate coefficients cannot be identified from those runs.

## A $2^2$ Design in Two Blocks

Suppose A and B each have low/high levels, but only two runs fit in a block. Use the AB product column to form blocks:

| Run | A | B | AB | Block |
|---|---:|---:|---:|---|
| $(1)$ | -1 | -1 | +1 | 1 |
| $a$ | +1 | -1 | -1 | 2 |
| $b$ | -1 | +1 | -1 | 2 |
| $ab$ | +1 | +1 | +1 | 1 |

Both A and B vary within each block. Their main effects remain orthogonal to the block contrast, but **AB is identical to that contrast**.

Writing $z=x_Ax_B$ for the coded block column,

$$\beta_{AB}x_Ax_B+\gamma z=(\beta_{AB}+\gamma)x_Ax_B.$$

Only the sum $\beta_{AB}+\gamma$ is identifiable. One cannot claim that an observed difference between blocks is specifically an AB interaction, or specifically a block effect.

:::{dropdown} Why Not Split by A or B?

Putting $(1),b$ in one block and $a,ab$ in the other makes every low-A run occur in one block and every high-A run in the other. A is then confounded with block.

Putting $(1),a$ together instead confounds B with block. Grouping by AB protects both main effects at the expense of the interaction. This choice is useful only if sacrificing AB is scientifically acceptable.
:::

## A $2^3$ Design in Two Blocks

A common choice is to confound the highest-order interaction ABC with block, protecting main effects and two-way interactions from the **additive** block effect.

| ABC sign | Runs |
|---|---|
| -1 | $(1),ab,ac,bc$ |
| +1 | $a,b,c,abc$ |

Each block contains four runs. Every main-effect and two-way sign column is balanced within a block.

Let $Z$ denote the block sign column. Since $Z=ABC$,

$$I=ABCZ.$$

Here $I$ is the all-$+1$ column, not an experimental run. The treatment label $(1)$ means all factors low; it is a different object.

### Multiplying Columns to Find Aliases

With $\pm1$ columns, each letter cancels when it occurs twice:

$$A^2=I,\qquad AB\cdot AC=BC.$$

Multiplying a defining relation by a term produces its alias. From $I=ABCZ$,

$$A=BCZ,\qquad AB=CZ,\qquad ABC=Z.$$

| Treatment term | Aliased term involving block |
|---|---|
| A | BCZ |
| B | ACZ |
| C | ABZ |
| AB | CZ |
| AC | BZ |
| BC | AZ |
| ABC | Z |

Under an additive block model, terms such as AZ and BCZ are assumed absent. If treatment effects change by block, these aliases matter: an estimated AB term can contain a C-by-block effect. Confounding with the highest-order treatment interaction does not protect against every possible block interaction.

:::{note}
Choose the blocking contrast using both design structure and process knowledge. “Use the highest-order interaction” is a common strategy, not a theorem that the sacrificed effect is negligible.
:::

## Degrees of Freedom & Error

For one unreplicated $2^3$ experiment split into two blocks, fitting the intercept, A, B, C, AB, AC, BC, and block uses all eight available columns. There is **no residual df**.

Blocking does not create replication or pure-error information. To make conventional significance tests, obtain independent error information through suitable replication or make explicit, defensible assumptions about negligible effects.

If only selected treatment terms are fitted, leftover variation may serve as model residual error, but it can include omitted treatment effects. It is not automatically pure error.

Randomize which physical block receives each block subset when feasible, then randomize run order within each block. Keep records of the nuisance conditions that motivated blocking.

## More Blocks: Include Products of the Generators

To divide a regular $2^k$ design into $2^q$ equal blocks, use $q$ independent block-generating columns. Their sign combinations identify the blocks.

The full block space includes **all nonidentity products** of the generators: $2^q-1$ columns. All corresponding treatment effects are confounded with block effects.

For a $2^3$ design in four blocks, choose $Z_1=AB$ and $Z_2=AC$. Their product is

$$Z_1Z_2=(AB)(AC)=BC.$$

Therefore AB, AC, and BC—not just the first two—are confounded with blocks.

| AB | AC | Runs in block |
|---:|---:|---|
| +1 | +1 | $(1),abc$ |
| +1 | -1 | $ab,c$ |
| -1 | +1 | $b,ac$ |
| -1 | -1 | $a,bc$ |

Each pair consists of opposite corners, so all three main effects vary within every block.

:::{dropdown} A Bad Generator Combination

Suppose the generators were AB and ABC. Their product is

$$AB\cdot ABC=C.$$

Even though neither chosen generator is a main effect, their product is. C would be confounded with a block contrast. Always inspect the complete block-generating group, not only the generators you wrote first.
:::

## Partial Confounding Across Replications

Repeating the same split can improve precision for estimable effects, but an interaction confounded with blocks in every replication stays confounded with the corresponding block contrasts.

**Partial confounding** changes the block generator across complete factorial replications:

| Replication | Interaction assigned to the block contrast |
|---|---|
| 1 | ABC |
| 2 | AB |
| 3 | AC |
| 4 | BC |

Each listed interaction is confounded in one replication and available from within-block comparisons in the other three. Main effects remain unconfounded with additive blocks throughout.

Analyze the actual structure, including replication and blocks within replication. Do not average all raw treatment responses as if they came from a completely randomized experiment. Different effects can have different amounts of information because they are recovered from different subsets of the runs.

:::{dropdown} Why Replication Alone Does Not Remove Aliasing

If the ABC block split is used four times, within each replicate the ABC column is still the block-contrast column. With separate block effects allowed in each replicate, an ABC effect remains inseparable from those contrasts.

Changing the split is what creates within-block information about ABC elsewhere. Additional measurements help only when the design supplies the missing distinction.
:::

## Check the Block Assignment in Python

This cell builds the factorial table, verifies block balance, and lists the four-block groups. It does not manufacture an error estimate for an unreplicated design.

In [1]:
import numpy as np
from collections import defaultdict

# Standard order: (1), a, b, ab, c, ac, bc, abc.
labels = ["(1)", "a", "b", "ab", "c", "ac", "bc", "abc"]
base = np.array([[1 if (i >> j) & 1 else -1 for j in range(3)]
                 for i in range(8)])
A, B, C = base.T
block = A*B*C
protected = np.column_stack([A, B, C, A*B, A*C, B*C])
assert np.all(protected.T @ block == 0)
for sign in [-1, 1]:
    print(f"ABC={sign:+}: {[labels[i] for i in range(8) if block[i] == sign]}")
    assert np.all(protected[block == sign].sum(axis=0) == 0)

four_blocks = defaultdict(list)
for name, ab, ac in zip(labels, A*B, A*C):
    four_blocks[(int(ab), int(ac))].append(name)
for key in sorted(four_blocks, reverse=True):
    print(f"(AB, AC)={key}: {four_blocks[key]}")
assert np.array_equal((A*B)*(A*C), B*C)
X = np.column_stack([np.ones(8), protected, block])
print("Two-block model rank:", np.linalg.matrix_rank(X), "; residual df:", 8-np.linalg.matrix_rank(X))


ABC=-1: ['(1)', 'ab', 'ac', 'bc']
ABC=+1: ['a', 'b', 'c', 'abc']
(AB, AC)=(1, 1): ['(1)', 'abc']
(AB, AC)=(1, -1): ['ab', 'c']
(AB, AC)=(-1, 1): ['b', 'ac']
(AB, AC)=(-1, -1): ['a', 'bc']
Two-block model rank: 8 ; residual df: 0


## Quick Review

| Question | Answer |
|---|---|
| Are fewer treatment combinations run? | No; the full set is distributed among blocks. |
| What is sacrificed? | Treatment contrasts identical to block contrasts. |
| Why multiply block generators? | Their products identify additional confounded effects. |
| Does randomization remove structural aliasing? | No; it protects assignment, but does not change matrix rank. |
| How can an interaction be recovered across replications? | Use a design where it is not confounded in every replication. |